# Analisis Sentimen Reputasi Undiksha dengan Hybrid Ensemble
### Multi-Platform (Twitter/X, YouTube, Instagram, TikTok) Menggunakan Leksikon InSet, Machine Learning Klasik, dan IndoBERT

> **Repositori Proyek Magang UPT TIK Undiksha**  
> **Penulis**: Mahasiswa Magang Ilmu Komputer  
> **Target Runtime**: Google Colab (Akselerator GPU T4)  
> **Prinsip Utama**: Reproducible (Seed 42), Anti-Leakage (Split sebelum feature extraction/SMOTE), dan Idempoten (Penyimpanan ke Google Drive).

---
### Panduan Penggunaan Singkat:
1. Pastikan runtime telah disetel ke **GPU T4** (*Runtime* > *Change runtime type* > *T4 GPU*).
2. Letakkan file dataset riil Anda di Google Drive pada folder:
   `MyDrive/analisis_sentimen_undiksha/data/undiksha_comments_combined.csv`  
   *(Jika file belum ada di Google Drive, Sel 1.1 akan otomatis memunculkan tombol upload agar Anda dapat mengunggahnya langsung dari laptop).*
3. Notebook dirancang untuk dijalankan berurutan dari sel teratas hingga terbawah (*Runtime* > *Run all*).
4. Semua hasil tabel (CSV), grafik (PNG 300 DPI), dan model otomatis disimpan ke Google Drive pada folder `MyDrive/analisis_sentimen_undiksha/`.

---
## Bagian 0: Setup Lingkungan, Dependensi, dan Konfigurasi Global

Pada bagian ini, sistem akan:
1. Memeriksa ketersediaan akselerator GPU (NVIDIA T4).
2. Melakukan *mount* Google Drive dan membuat struktur direktori kerja otomatis.
3. Memasang seluruh pustaka Python yang diperlukan.
4. Mengatur nilai *seed* acak global untuk menjamin reproduktifitas eksperimen.
5. Menetapkan parameter konfigurasi terpusat (tanpa hardcode di sel lain).

In [ ]:
# Sel 0.1: Pemeriksaan GPU dan Mount Google Drive
import os
import sys
import subprocess

# Periksa GPU
try:
    res = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if res.returncode == 0:
        print("[GPU INFO]\n" + res.stdout[:400] + "...")
    else:
        print("[INFO] GPU tidak terdeteksi via nvidia-smi, berjalan di mode CPU.")
except Exception as e:
    print("[INFO] Pemeriksaan GPU dilewati:", e)

# Mount Google Drive jika berjalan di lingkungan Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    print("[INFO] Berjalan di Google Colab. Melakukan mount Google Drive...")
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/analisis_sentimen_undiksha'
else:
    print("[INFO] Berjalan di lingkungan lokal. Menggunakan direktori kerja lokal...")
    BASE_DIR = os.path.abspath('./drive_undiksha')

# Buat struktur direktori kerja terstandar
DIRS = {
    'base': BASE_DIR,
    'data': os.path.join(BASE_DIR, 'data'),
    'models': os.path.join(BASE_DIR, 'models'),
    'reports': os.path.join(BASE_DIR, 'reports'),
    'figures': os.path.join(BASE_DIR, 'reports', 'figures'),
    'tables': os.path.join(BASE_DIR, 'reports', 'tables'),
    'checkpoints': os.path.join(BASE_DIR, 'checkpoints')
}

for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)
    print(f"Direktori siap: {name.upper()} -> {path}")

In [ ]:
# Sel 0.2: Instalasi Dependensi Pustaka
# Menginstall Sastrawi, emoji, imbalanced-learn, transformers, datasets, accelerate, wordcloud, statsmodels
!pip install -q PySastrawi emoji imbalanced-learn transformers datasets accelerate wordcloud statsmodels scikit-learn
print("[OK] Seluruh dependensi berhasil dipasang!")

In [ ]:
# Sel 0.3: Import Library dan Pengaturan Global Seed
import time
import json
import re
import math
import random
import pickle
import warnings
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, f1_score
)
from sklearn.utils.class_weight import compute_class_weight

import transformers
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)

warnings.filterwarnings('ignore')

# Pengaturan visualisasi
sns.set_theme(style="whitegrid", font="DejaVu Sans")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300

def set_seed(seed=42):
    """Menetapkan seed acak pada seluruh pustaka agar hasil sepenuhnya reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"[SEED] Seed acak global ditetapkan ke: {seed}")

set_seed(42)

In [ ]:
# Sel 0.4: Konfigurasi Terpusat Proyek (CONFIG)
# CATATAN: Semua parameter ditetapkan di sini; dilarang hardcode di sel bawah.
CONFIG = {
    # Pengaturan Eksperimen & Data Riil
    'SEED': 42,
    'DATA_CSV_PATH': os.path.join(DIRS['data'], 'undiksha_comments_combined.csv'),
    
    # Skema Label Sentimen
    'LABEL_MAP': {'negatif': 0, 'netral': 1, 'positif': 2},
    'LABEL_NAMES': ['negatif', 'netral', 'positif'],
    'TARGET_COL': 'sentimen',
    'TEXT_RAW_COL': 'teks',
    'TEXT_CLEAN_COL': 'teks_bersih',
    
    # Rasio Pembagian Data (Stratified Split)
    'SPLIT_RATIO': {'train': 0.70, 'val': 0.15, 'test': 0.15},
    
    # Penanganan Ketidakseimbangan Data
    'USE_CLASS_WEIGHTS': True,
    'USE_SMOTE': False,  # Opsi SMOTE untuk TF-IDF data latih
    
    # Parameter TF-IDF
    'TFIDF_MAX_FEATURES': 5000,
    'TFIDF_NGRAM_RANGE': (1, 2),
    
    # Parameter IndoBERT
    'BERT_MODEL_NAME': 'indobenchmark/indobert-base-p1',  # Opsi: 'indolem/indobertweet-base-uncased'
    'MAX_LENGTH': 128,
    'BATCH_SIZE': 16 if torch.cuda.is_available() else 4,
    'GRADIENT_ACCUMULATION_STEPS': 2,
    'EPOCHS': 3,
    'LEARNING_RATE': 2e-5,
    'WARMUP_RATIO': 0.1,
    'WEIGHT_DECAY': 0.01,
    
    # 7 Aspek Reputasi Undiksha
    'ASPECTS': {
        'akademik': ['dosen', 'kuliah', 'kurikulum', 'akreditasi', 'skripsi', 'jurusan', 'prodi', 'kelas', 'ujian', 'praktikum', 'krs', 'matkul', 'pengajar', 'belajar'],
        'penerimaan_biaya': ['ukt', 'biaya', 'bayar', 'snbp', 'snbt', 'mandiri', 'beasiswa', 'kip', 'pendaftaran', 'uang', 'mahal', 'murah', 'registrasi', 'briva', 'cicilan'],
        'fasilitas': ['kampus', 'gedung', 'lab', 'wifi', 'internet', 'toilet', 'parkir', 'perpustakaan', 'ruang', 'ac', 'proyektor', 'asrama', 'kos', 'sarana'],
        'layanan_administrasi': ['siakad', 'birokrasi', 'surat', 'legalisir', 'staf', 'pegawai', 'admin', 'pelayanan', 'antre', 'lama', 'lambat', 'ramah', 'respons', 'cs'],
        'kemahasiswaan': ['bem', 'hima', 'ukm', 'organisasi', 'prestasi', 'lomba', 'kkn', 'orasi', 'demo', 'kegiatan', 'event', 'expo', 'juara', 'seminar'],
        'lulusan_karier': ['wisuda', 'alumni', 'kerja', 'lulus', 'karir', 'ijazah', 'lowongan', 'tracer', 'toga', 'ipk', 'cumlaude'],
        'citra_umum': ['undiksha', 'singaraja', 'bali', 'rektor', 'berita', 'viral', 'kebijakan', 'bangga', 'reputasi', 'kampusku', 'ganesha', 'tercinta']
    }
}

print("[OK] Konfigurasi global data riil berhasil dimuat.")

---
## Bagian 1: Pemuatan Dataset Riil dan Validasi Skema

Bagian ini membaca dataset sentimen opini publik kampus Undiksha dari Google Drive.
Standar skema kolom yang didukung:
* `id_anonim` (atau `id_anon`): Penanda unik anonim komentar (menjaga privasi UU PDP).
* `platform`: Sumber media sosial (`twitter`, `youtube`, `instagram`, `tiktok`).
* `teks` (atau `text`/`text_raw`): Opini mentah masyarakat / mahasiswa.
* `tanggal` (atau `created_at`): Waktu unggahan (YYYY-MM-DD).
* `sentimen` (atau `sentiment`): Label polaritas (`positif`, `netral`, `negatif`).
* `is_official`: Boolean penanda akun resmi universitas (agar dapat dipisahkan saat analisis reputasi).

Jika file belum tersedia di Google Drive, sel ini secara otomatis menyediakan tombol unggah file langsung dari komputer Anda.

In [ ]:
# Sel 1.1: Pemuatan Dataset Riil Undiksha dari Google Drive
csv_path = CONFIG['DATA_CSV_PATH']

# Cek apakah file sudah ada di Google Drive
if not os.path.exists(csv_path):
    # Periksa apakah ada file CSV lain di folder data
    alt_files = [f for f in os.listdir(DIRS['data']) if f.endswith('.csv') and 'idx' not in f]
    if alt_files:
        csv_path = os.path.join(DIRS['data'], alt_files[0])
        print(f"[INFO] Menggunakan file CSV yang tersedia di folder data: {csv_path}")
    else:
        if IN_COLAB:
            print(f"[INFO] Dataset belum ditemukan di: {CONFIG['DATA_CSV_PATH']}")
            print("[PANDUAN] Silakan klik tombol 'Choose Files' di bawah ini untuk mengunggah file CSV dataset riil Anda (misal: combined_labeled.csv):")
            from google.colab import files
            uploaded = files.upload()
            if uploaded:
                uploaded_name = list(uploaded.keys())[0]
                import shutil
                shutil.move(uploaded_name, CONFIG['DATA_CSV_PATH'])
                csv_path = CONFIG['DATA_CSV_PATH']
                print(f"[OK] File berhasil diunggah dan disimpan ke: {csv_path}")
        else:
            local_default = 'data/labeled/combined_labeled.csv'
            if os.path.exists(local_default):
                csv_path = local_default
                print(f"[INFO LOKAL] Menggunakan dataset lokal: {local_default}")
            else:
                raise FileNotFoundError(
                    f"Dataset riil tidak ditemukan di {CONFIG['DATA_CSV_PATH']}. "
                    f"Silakan letakkan file dataset di Google Drive pada MyDrive/analisis_sentimen_undiksha/data/undiksha_comments_combined.csv"
                )

print(f"[INFO] Membaca dataset dari: {csv_path}")
df = pd.read_csv(csv_path)

# Standarisasi nama kolom otomatis (kompatibel skema Inggris maupun Indonesia)
col_rename = {
    'id_anon': 'id_anonim',
    'text': 'teks',
    'text_raw': 'teks',
    'sentiment': 'sentimen',
    'created_at': 'tanggal',
    'date': 'tanggal'
}
df = df.rename(columns={k: v for k, v in col_rename.items() if k in df.columns})

# Validasi skema kolom wajib
wajib_kolom = ['id_anonim', 'platform', 'teks', 'sentimen']
for col in wajib_kolom:
    if col not in df.columns:
        raise ValueError(f"Kolom wajib '{col}' tidak ditemukan di dataset! Kolom yang ada: {list(df.columns)}")

# Tambahkan kolom default jika belum ada
if 'tanggal' not in df.columns:
    df['tanggal'] = '2026-01-01'
if 'is_official' not in df.columns:
    df['is_official'] = False

# Hapus baris dengan teks kosong atau duplikat
baris_awal = len(df)
df = df.dropna(subset=['teks', 'sentimen'])
df['teks'] = df['teks'].astype(str).str.strip()
df = df[df['teks'] != '']
df = df.drop_duplicates(subset=['teks'])
baris_akhir = len(df)

print(f"\nTotal baris dimuat : {baris_awal}")
print(f"Total baris bersih: {baris_akhir} (dihapus {baris_awal - baris_akhir} duplikat/kosong)\n")
print("Contoh 5 Baris Data Teratas:")
display(df.head(5))

---
## Bagian 2: Analisis Data Eksploratif (EDA)

Bagian ini memetakan karakteristik mendasar dari dataset:
1. Distribusi polaritas sentimen keseluruhan dan rincian per platform media sosial.
2. Panjang sebaran karakter dan jumlah kata dalam opini.
3. Proporsi postingan dari akun resmi institusi vs masyarakat umum.
4. Kata-kata yang paling dominan muncul pada tiap kelas sentimen.

In [ ]:
# Sel 2.1: Analisis Distribusi Sentimen per Platform
df['panjang_karakter'] = df['teks'].apply(len)
df['jumlah_kata'] = df['teks'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribusi Keseluruhan
palette = {'positif': '#2ecc71', 'netral': '#3498db', 'negatif': '#e74c3c'}
sns.countplot(data=df, x='sentimen', order=['positif', 'netral', 'negatif'], palette=palette, ax=axes[0])
axes[0].set_title("Distribusi Sentimen Keseluruhan", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Kelas Sentimen")
axes[0].set_ylabel("Jumlah Komentar")
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha='center', va='center', color='white', fontweight='bold')

# Plot 2: Distribusi per Platform
sns.countplot(data=df, x='platform', hue='sentimen', palette=palette, ax=axes[1])
axes[1].set_title("Distribusi Sentimen Lintas 4 Platform Media Sosial", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Platform")
axes[1].set_ylabel("Jumlah Komentar")
axes[1].legend(title="Sentimen")

plt.tight_layout()
plt.savefig(os.path.join(DIRS['figures'], 'eda_distribusi_sentimen.png'))
plt.show()

# Ringkasan statistik
tabel_eda = pd.crosstab(df['platform'], df['sentimen'], margins=True)
print("Tabel Silang Platform vs Sentimen:")
display(tabel_eda)

In [ ]:
# Sel 2.2: Analisis Panjang Teks dan Akun Resmi
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Distribusi Panjang Teks
sns.histplot(data=df, x='jumlah_kata', hue='sentimen', palette=palette, kde=True, ax=axes[0], bins=25)
axes[0].set_title("Distribusi Jumlah Kata per Sentimen", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Jumlah Kata")
axes[0].set_ylabel("Frekuensi")

# Proporsi Akun Resmi vs Akun Publik
sns.countplot(data=df, x='is_official', hue='sentimen', palette=palette, ax=axes[1])
axes[1].set_title("Proporsi Sentimen pada Akun Resmi vs Publik", fontsize=12, fontweight='bold')
axes[1].set_xticklabels(['Akun Publik (False)', 'Akun Resmi (True)'])
axes[1].set_xlabel("Status Akun Resmi")
axes[1].set_ylabel("Jumlah")

plt.tight_layout()
plt.savefig(os.path.join(DIRS['figures'], 'eda_panjang_dan_official.png'))
plt.show()

print(f"Rata-rata panjang kata: {df['jumlah_kata'].mean():.2f} kata (min: {df['jumlah_kata'].min()}, max: {df['jumlah_kata'].max()})")
print(f"Persentase Akun Resmi : {(df['is_official'].sum() / len(df) * 100):.2f}%")

---
## Bagian 3: Pipeline Preprocessing Teks

Tahapan pembersihan teks dirancang terstruktur dan berurutan sesuai pedoman proyek:
1. **Case Folding**: Mengubah seluruh teks menjadi huruf kecil (*lowercase*).
2. **Pembersihan Entitas Khusus**: Menghapus URL, tautan web, dan mention akun (`@username`).
3. **Normalisasi Hashtag**: Menghilangkan simbol tagar namun tetap mempertahankan kata di dalamnya (`#undikshajaya` -> `undikshajaya`).
4. **Emoji Parsing**: Mengonversi simbol emoji menjadi teks bermakna bahasa Indonesia (misal: `❤️` menjadi `cinta`, `😡` menjadi `marah`).
5. **Normalisasi Slang & Singkatan**: Mengubah bahasa gaul dan singkatan mahasiswa ke bentuk baku bahasa Indonesia menggunakan kamus CSV/Dictionary yang dapat diperluas.
6. **Pembersihan Noise**: Menghilangkan tanda baca dan karakter khusus non-alfanumerik.
7. **Stopwords Removal Selektif**: Menghilangkan kata sambung umum tanpa membuang kata negasi krusial (`tidak`, `belum`, `jangan`, `kurang`).
8. **Stemming Sastrawi dengan Cache**: Mereduksi kata berimbuhan ke kata dasar dengan memanfaatkan `lru_cache` agar eksekusi berlangsung cepat.

In [ ]:
# Sel 3.1: Definisi Kamus Slang, Emoji, dan Stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import emoji

# Inisialisasi Stemmer Sastrawi dengan Cache Memori
stemmer_factory = StemmerFactory()
_sastrawi_stemmer = stemmer_factory.create_stemmer()

@lru_cache(maxsize=15000)
def cached_stem(word):
    """Stemming kata dengan LRU cache untuk mempercepat komputasi berulang."""
    return _sastrawi_stemmer.stem(word)

# Kamus Slang Dasar yang dapat diperluas
SLANG_DICT = {
    'bgt': 'banget', 'bngt': 'banget', 'bgtt': 'banget',
    'ga': 'tidak', 'gak': 'tidak', 'nggak': 'tidak', 'ngga': 'tidak', 'g': 'tidak', 'tak': 'tidak',
    'bgs': 'bagus', 'bagoes': 'bagus', 'mantul': 'mantap betul', 'gokil': 'hebat',
    'lemot': 'lambat', 'lola': 'lambat', 'ancur': 'hancur', 'parah': 'buruk',
    'k登陆': 'krs', 'kmps': 'kampus', 'mhs': 'mahasiswa', 'dospem': 'dosen pembimbing',
    'doswal': 'dosen wali', 'matkul': 'mata kuliah', 'perpus': 'perpustakaan',
    'recs': 'rekomendasi', 'info': 'informasi', 'min': 'admin', 'makasi': 'terima kasih',
    'thx': 'terima kasih', 'thanks': 'terima kasih', 'makasih': 'terima kasih',
    'krn': 'karena', 'karna': 'karena', 'dgn': 'dengan', 'sdh': 'sudah', 'udh': 'sudah',
    'blm': 'belum', 'blom': 'belum', 'tp': 'tetapi', 'tpi': 'tetapi', 'jg': 'juga',
    'jgn': 'jangan', 'jd': 'jadi', 'jdi': 'jadi', 'bisa': 'bisa', 'bs': 'bisa',
    'bener': 'benar', 'bnr': 'benar', 'bgtu': 'begitu', 'knp': 'kenapa', 'kenapah': 'kenapa',
    'becik': 'bagus', 'sing': 'tidak', 'ngidang': 'bisa', 'suksma': 'terima kasih' # Istilah Bali
}

# Simpan kamus slang ke Drive agar dapat dimodifikasi secara eksternal
slang_path = os.path.join(DIRS['data'], 'slang_dict.json')
with open(slang_path, 'w', encoding='utf-8') as f:
    json.dump(SLANG_DICT, f, indent=2, ensure_ascii=False)

# Stopwords selektif: jangan hapus kata negasi penting!
stop_factory = StopWordRemoverFactory()
default_stopwords = set(stop_factory.get_stop_words())
NEGATION_WORDS = {'tidak', 'bukan', 'jangan', 'belum', 'kurang', 'tanpa'}
CUSTOM_STOPWORDS = default_stopwords - NEGATION_WORDS

# Kamus terjemahan emoji umum ke teks ekspresi
EMOJI_DICT = {
    '❤️': ' cinta ', '😍': ' suka sekali ', '👍': ' bagus ', '🔥': ' keren ',
    '😭': ' sedih menangis ', '😡': ' marah ', '😠': ' kesal ', '👎': ' jelek buruk ',
    '🙏': ' terima kasih tolong ', '👏': ' hebat salut ', '🤣': ' tertawa '
}

In [ ]:
# Sel 3.2: Pipeline Preprocessing Lengkap
def clean_text_pipeline(text, use_stemming=True):
    """Menjalankan seluruh tahapan pra-pemrosesan teks bahasa Indonesia."""
    if not isinstance(text, str) or not text.strip():
        return ""
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Hapus URL & Tautan
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    
    # 3. Hapus Mention pengguna (@username)
    text = re.sub(r'@\w+', ' ', text)
    
    # 4. Normalisasi Hashtag (hapus tanda pagar, pertahankan kata)
    text = re.sub(r'#(\w+)', r'\1', text)
    
    # 5. Emoji to Text ekspresi
    for emo, rep in EMOJI_DICT.items():
        text = text.replace(emo, rep)
    text = emoji.demojize(text, language='id') if hasattr(emoji, 'demojize') else text
    text = re.sub(r':[a-z_]+:', ' ', text)  # Bersihkan sisa demojize tag
    
    # 6. Pembersihan Karakter Khusus & Tanda Baca (simpan huruf dan angka)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # 7. Normalisasi Slang & Singkatan
    tokens = text.split()
    tokens = [SLANG_DICT.get(t, t) for t in tokens]
    
    # 8. Stopwords Removal Selektif
    tokens = [t for t in tokens if t not in CUSTOM_STOPWORDS and len(t) > 1]
    
    # 9. Stemming Sastrawi Berbasis Cache
    if use_stemming:
        tokens = [cached_stem(t) for t in tokens]
        
    cleaned = " ".join(tokens)
    # Hapus spasi berlebih
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

# Terapkan pada dataset
t0 = time.time()
print("[INFO] Menjalankan preprocessing teks pada seluruh dataset...")
df['teks_bersih'] = df['teks'].apply(lambda x: clean_text_pipeline(x, use_stemming=True))
print(f"[OK] Selesai dalam {time.time() - t0:.2f} detik.")

# Filter baris yang mungkin kosong setelah pembersihan
df = df[df['teks_bersih'].str.strip() != '']

# Tampilkan 10 Sampel Perbandingan Sebelum vs Sesudah
print("\n--- Tabel 10 Contoh Perbandingan Teks Asli vs Teks Bersih ---")
sample_preview = df[['teks', 'teks_bersih', 'sentimen']].head(10)
display(sample_preview)

---
## Bagian 4: Pembagian Data Terstratifikasi (Anti-Leakage)

Untuk memastikan pengujian model bersifat objektif dan tidak mengalami kebocoran data (*data leakage*):
1. Dataset dibagi menjadi **Train Set (70%)**, **Validation Set (15%)**, dan **Test Set (15%)**.
2. Pembagian dilakukan secara terstratifikasi (*Stratified Split*) terhadap kombinasi kelas label dan platform.
3. Seluruh proses pembelajaran representasi (TF-IDF, penanganan ketidakseimbangan, fine-tuning) **hanya dilakukan pada Train Set**.
4. Indeks pembagian disimpan ke Google Drive (`train_idx.csv`, `val_idx.csv`, `test_idx.csv`) agar eksperimen dapat direproduksi persis kapan pun.

In [ ]:
# Sel 4.1: Eksekusi Stratified Train-Validation-Test Split
# Bentuk strata gabungan untuk menjaga proporsi sentimen dan platform
strata = df['sentimen'].astype(str) + "_" + df['platform'].astype(str)

# Tahap 1: Pisahkan Test Set (15%)
train_val_df, test_df = train_test_split(
    df,
    test_size=CONFIG['SPLIT_RATIO']['test'],
    stratify=strata,
    random_state=CONFIG['SEED']
)

# Tahap 2: Pisahkan Train (70% total) dan Validation (15% total)
# 15% dari total = 15/85 (~0.1765) dari sisa train_val
val_relative_ratio = CONFIG['SPLIT_RATIO']['val'] / (CONFIG['SPLIT_RATIO']['train'] + CONFIG['SPLIT_RATIO']['val'])
strata_train_val = train_val_df['sentimen'].astype(str) + "_" + train_val_df['platform'].astype(str)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_relative_ratio,
    stratify=strata_train_val,
    random_state=CONFIG['SEED']
)

# Reset Index
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Simpan indeks split ke Drive
train_df[['id_anonim']].to_csv(os.path.join(DIRS['data'], 'train_idx.csv'), index=False)
val_df[['id_anonim']].to_csv(os.path.join(DIRS['data'], 'val_idx.csv'), index=False)
test_df[['id_anonim']].to_csv(os.path.join(DIRS['data'], 'test_idx.csv'), index=False)

print(f"Jumlah Data Latih (Train)     : {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Jumlah Data Validasi (Val)    : {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Jumlah Data Uji (Test)        : {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")
print(f"Total Sampel                  : {len(df)}")

# Verifikasi proporsi kelas di tiap himpunan
split_summary = pd.DataFrame({
    'Train (%)': train_df['sentimen'].value_counts(normalize=True) * 100,
    'Val (%)': val_df['sentimen'].value_counts(normalize=True) * 100,
    'Test (%)': test_df['sentimen'].value_counts(normalize=True) * 100
})
print("\nProporsi Kelas Sentimen Lintas Partisi Data:")
display(split_summary.round(2))

---
## Bagian 5: Penanganan Ketidakseimbangan Kelas (Class Imbalance)

Opini publik kampus di media sosial umumnya didominasi oleh kelas **Netral** (informasi pengumuman, pertanyaan umum), sedangkan kelas **Negatif** (keluhan kritis) memiliki jumlah sampel yang lebih sedikit.

Strategi penanganan yang diterapkan:
1. **Perhitungan Bobot Kelas Terbalik (*Inverse Class Weights*)**: Memberikan penalti loss yang lebih tinggi saat model salah mengklasifikasikan kelas minoritas pada SVM dan IndoBERT Cross-Entropy Loss.
2. **SMOTE Opsional pada Data Latih**: Mencegah kebocoran data dengan menerapkan SMOTE hanya pada matriks fitur TF-IDF data latih.

In [ ]:
# Sel 5.1: Perhitungan Class Weights
y_train_raw = train_df['sentimen'].values
unique_classes = np.array(['negatif', 'netral', 'positif'])

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=y_train_raw
)

CLASS_WEIGHTS_DICT = {cls: weight for cls, weight in zip(unique_classes, class_weights)}
# Versi Tensor untuk PyTorch / IndoBERT Loss Function
CLASS_WEIGHTS_TENSOR = torch.tensor(
    [CLASS_WEIGHTS_DICT[c] for c in ['negatif', 'netral', 'positif']],
    dtype=torch.float
)
if torch.cuda.is_available():
    CLASS_WEIGHTS_TENSOR = CLASS_WEIGHTS_TENSOR.cuda()

print("Bobot Kelas Terbobot (Class Weights):")
for cls, w in CLASS_WEIGHTS_DICT.items():
    print(f" - {cls:<8}: {w:.4f}")

---
## Bagian 6: Komponen A – Pendekatan Berbasis Leksikon (InSet Lexicon)

**InSet (Indonesian Sentiment Lexicon)** memuat ribuan kata positif dan negatif berbahasa Indonesia dengan bobot polaritas:
* Fitur polaritas dihitung berdasarkan frekuensi kata positif vs negatif dalam teks.
* Dilengkapi algoritma *negation window* (membalik polaritas kata positif apabila didahului kata penyangkal seperti `tidak`, `kurang`, `belum`, `jangan`).
* Mengonversi skor polaritas menjadi distribusi probabilitas 3 kelas `[P(neg), P(net), P(pos)]` menggunakan fungsi softmax.

In [ ]:
# Sel 6.1: Pemuatan dan Algoritma Leksikon InSet
# Kamus leksikon InSet dasar (dilengkapi kosa kata kontekstual institusi)
INSET_POSITIVE = {
    'bagus': 4, 'keren': 4, 'hebat': 4, 'unggul': 5, 'ramah': 3, 'cepat': 3,
    'nyaman': 4, 'suka': 3, 'cinta': 4, 'bangga': 5, 'bersih': 3, 'lengkap': 3,
    'sejuk': 3, 'murah': 3, 'prestasi': 4, 'juara': 4, 'terbaik': 5, 'mantap': 4,
    'terima kasih': 3, 'bantu': 3, 'solutif': 4, 'sabar': 3, 'megah': 3, 'sukses': 4,
    'becik': 4, 'suksma': 4, 'maju': 3, 'senang': 3, 'puas': 4, 'mudah': 3
}

INSET_NEGATIVE = {
    'rusak': -4, 'jelek': -4, 'buruk': -4, 'lemot': -4, 'lambat': -4, 'kecewa': -5,
    'marah': -4, 'emosi': -4, 'error': -4, 'down': -4, 'hancur': -5, 'mahal': -3,
    'sulit': -3, 'susah': -3, 'antre': -3, 'cuek': -3, 'sempit': -3, 'semrawut': -3,
    'payah': -4, 'mati': -3, 'kotor': -3, 'parah': -4, 'rugi': -4, 'keluh': -3,
    'masalah': -3, 'ribet': -3, 'bobrok': -5, 'nyesel': -4, 'bohong': -4, 'sing': -2
}

NEGATION_SET = {'tidak', 'bukan', 'jangan', 'belum', 'kurang', 'tak'}

def calculate_lexicon_scores(text):
    """Menghitung skor polaritas positif dan negatif dengan penanganan negasi."""
    tokens = text.split()
    pos_score = 0.0
    neg_score = 0.0
    
    for i, token in enumerate(tokens):
        # Periksa apakah token didahului kata negasi (window 1-2 kata sebelumnya)
        is_negated = False
        if i > 0 and tokens[i-1] in NEGATION_SET:
            is_negated = True
        elif i > 1 and tokens[i-2] in NEGATION_SET:
            is_negated = True
            
        p_val = INSET_POSITIVE.get(token, 0)
        n_val = INSET_NEGATIVE.get(token, 0)
        
        if is_negated:
            # Jika ada negasi, balik polaritas
            if p_val > 0:
                neg_score += p_val
            elif n_val < 0:
                pos_score += abs(n_val)
        else:
            pos_score += p_val
            neg_score += abs(n_val)
            
    total_score = pos_score - neg_score
    
    # Softmax probabilitas 3 kelas: [negatif, netral, positif]
    # Netral diasumsikan memiliki prior base jika skor positif & negatif kecil
    raw_logits = np.array([neg_score, 1.5, pos_score])
    exp_logits = np.exp(raw_logits - np.max(raw_logits))
    probs = exp_logits / np.sum(exp_logits)
    
    pred_idx = np.argmax(probs)
    labels = ['negatif', 'netral', 'positif']
    return probs, labels[pred_idx]

# Evaluasi InSet pada Validation Set
val_lexicon_probs = []
val_lexicon_preds = []

for txt in val_df['teks_bersih']:
    p, l = calculate_lexicon_scores(txt)
    val_lexicon_probs.append(p)
    val_lexicon_preds.append(l)
    
val_lexicon_probs = np.array(val_lexicon_probs)
val_lexicon_acc = accuracy_score(val_df['sentimen'], val_lexicon_preds)
val_lexicon_f1 = f1_score(val_df['sentimen'], val_lexicon_preds, average='macro')

print(f"[KOMPONEN A - LEXICON InSet] Hasil pada Validation Set:")
print(f" -> Akurasi : {val_lexicon_acc * 100:.2f}%")
print(f" -> F1-Macro: {val_lexicon_f1 * 100:.2f}%")

---
## Bagian 7: Komponen B – Machine Learning Klasik (TF-IDF + ML)

Pada tahapan ini:
1. `TfidfVectorizer` (unigram + bigram, hingga 5.000 fitur) di-fit **hanya pada Train Set**, lalu mentransformasikan Data Latih, Validasi, dan Uji.
2. Melatih 3 algoritma pembelajaran mesin terawasi:
   * **Support Vector Machine (SVM RBF Kernel)** dengan `probability=True` dan `class_weight='balanced'`.
   * **Multinomial Naive Bayes (MNB)** dengan smoothing Laplacian $\alpha=0.5$.
   * **Random Forest (RF)** dengan 100 pohon keputusan dan pembobotan kelas seimbang.
3. Memilih model ML klasik dengan performa terbaik pada Validation Set untuk diteruskan ke tahap Ensemble.

In [ ]:
# Sel 7.1: Ekstraksi Fitur TF-IDF & Pelatihan Model Klasik
# Fitting TF-IDF Vectorizer HANYA pada Train Set
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=CONFIG['TFIDF_NGRAM_RANGE'],
    max_features=CONFIG['TFIDF_MAX_FEATURES']
)

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['teks_bersih'])
X_val_tfidf = tfidf_vectorizer.transform(val_df['teks_bersih'])
X_test_tfidf = tfidf_vectorizer.transform(test_df['teks_bersih'])

y_train_num = train_df['sentimen'].map(CONFIG['LABEL_MAP']).values
y_val_num = val_df['sentimen'].map(CONFIG['LABEL_MAP']).values
y_test_num = test_df['sentimen'].map(CONFIG['LABEL_MAP']).values

# Inisialisasi 3 Model Pembelajaran Mesin
classical_models = {
    'SVM_RBF': SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=CONFIG['SEED']),
    'Naive_Bayes': MultinomialNB(alpha=0.5),
    'Random_Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=CONFIG['SEED'])
}

ml_results = {}
val_ml_probs = {}

for name, model in classical_models.items():
    t_start = time.time()
    model.fit(X_train_tfidf, y_train_num)
    t_train = time.time() - t_start
    
    # Prediksi probabilitas pada Validation Set
    probs = model.predict_proba(X_val_tfidf)
    preds = np.argmax(probs, axis=1)
    
    acc = accuracy_score(y_val_num, preds)
    f1 = f1_score(y_val_num, preds, average='macro')
    
    ml_results[name] = {'Akurasi': acc, 'F1-Macro': f1, 'Waktu (s)': t_train, 'model_obj': model}
    val_ml_probs[name] = probs
    print(f"[{name:<13}] Latih: {t_train:.2f}s | Val Akurasi: {acc*100:.2f}% | Val F1-Macro: {f1*100:.2f}%")

# Pilih model ML klasik terbaik berdasarkan F1-Macro
best_ml_name = max(ml_results.keys(), key=lambda k: ml_results[k]['F1-Macro'])
best_ml_model = ml_results[best_ml_name]['model_obj']
print(f"\n[TERPILIH] Model ML Klasik Terbaik: {best_ml_name} (F1: {ml_results[best_ml_name]['F1-Macro']*100:.2f}%)")

# Simpan TF-IDF dan Model ML Terbaik ke Drive
with open(os.path.join(DIRS['models'], 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
with open(os.path.join(DIRS['models'], 'best_classical_model.pkl'), 'wb') as f:
    pickle.dump(best_ml_model, f)
print(f"[OK] TF-IDF dan {best_ml_name} berhasil disimpan ke Drive.")

---
## Bagian 8: Komponen C – Fine-Tuning Transformer (IndoBERT)

**IndoBERT (`indobenchmark/indobert-base-p1`)** adalah model bahasa berbasis Transformer yang telah di-pra-latih pada korpus bahasa Indonesia berskala besar (Indo4B):
* Menghasilkan representasi kontekstual yang mendalam terhadap struktur kalimat, sinonim, dan ambiguitas kata.
* Dilatih dengan akselerasi FP16 (pada GPU T4), *early stopping* (patience=2 epoch), dan *weighted cross-entropy loss* untuk mencegah bias pada kelas mayoritas.
* Checkpoint terbaik secara otomatis disimpan ke Google Drive.

In [ ]:
# Sel 8.1: Tokenisasi Dataset & Custom PyTorch Dataset
tokenizer = AutoTokenizer.from_pretrained(CONFIG['BERT_MODEL_NAME'])

class UndikshaSentimentDataset(Dataset):
    """Dataset PyTorch untuk tokenisasi batch input Transformer IndoBERT."""
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Bentuk Train, Val, dan Test Datasets
train_dataset = UndikshaSentimentDataset(train_df['teks_bersih'], y_train_num, tokenizer, CONFIG['MAX_LENGTH'])
val_dataset = UndikshaSentimentDataset(val_df['teks_bersih'], y_val_num, tokenizer, CONFIG['MAX_LENGTH'])
test_dataset = UndikshaSentimentDataset(test_df['teks_bersih'], y_test_num, tokenizer, CONFIG['MAX_LENGTH'])

print(f"[OK] Dataset IndoBERT siap. Max sequence length: {CONFIG['MAX_LENGTH']}")

In [ ]:
# Sel 8.2: Custom Weighted Trainer & Fine-Tuning IndoBERT
from transformers import Trainer

class WeightedLossTrainer(Trainer):
    """Trainer dengan modifikasi Loss Function CrossEntropy terbobot (Class Weights)."""
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS_TENSOR)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_bert_metrics(eval_pred):
    """Menghitung metrik Akurasi dan F1-Macro selama evaluasi Trainer."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'f1_macro': f1}

bert_output_dir = os.path.join(DIRS['checkpoints'], 'indobert_undiksha')

import inspect
train_args_kwargs = {
    'output_dir': bert_output_dir,
    'num_train_epochs': CONFIG['EPOCHS'],
    'per_device_train_batch_size': CONFIG['BATCH_SIZE'],
    'per_device_eval_batch_size': CONFIG['BATCH_SIZE'] * 2,
    'gradient_accumulation_steps': CONFIG['GRADIENT_ACCUMULATION_STEPS'],
    'learning_rate': CONFIG['LEARNING_RATE'],
    'weight_decay': CONFIG['WEIGHT_DECAY'],
    'save_strategy': "epoch",
    'load_best_model_at_end': True,
    'metric_for_best_model': "f1_macro",
    'greater_is_better': True,
    'fp16': torch.cuda.is_available(),
    'logging_steps': 10,
    'report_to': "none",
    'seed': CONFIG['SEED']
}

# Deteksi kompatibilitas nama parameter versi transformers
sig = inspect.signature(TrainingArguments.__init__)
eval_key = 'eval_strategy' if 'eval_strategy' in sig.parameters else 'evaluation_strategy'
train_args_kwargs[eval_key] = "epoch"

training_args = TrainingArguments(**train_args_kwargs)

# Inisialisasi Model IndoBERT Pretrained
bert_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['BERT_MODEL_NAME'],
    num_labels=3
)
if torch.cuda.is_available():
    bert_model = bert_model.cuda()

trainer = WeightedLossTrainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_bert_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("[INFO] Memulai pelatihan Fine-Tuning IndoBERT...")
t0_bert = time.time()
trainer.train()
print(f"[OK] Pelatihan IndoBERT selesai dalam {(time.time() - t0_bert)/60:.2f} menit!")

# Simpan Model Terbaik dan Tokenizer ke Drive
bert_best_dir = os.path.join(DIRS['models'], 'indobert_best')
trainer.save_model(bert_best_dir)
tokenizer.save_pretrained(bert_best_dir)
print(f"[OK] Model IndoBERT terbaik tersimpan di: {bert_best_dir}")

In [ ]:
# Sel 8.3: Ekstraksi Probabilitas Prediksi IndoBERT
def get_bert_probabilities(model, dataset):
    """Ekstraksi probabilitas softmax dari output logits IndoBERT."""
    model.eval()
    all_probs = []
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=CONFIG['BATCH_SIZE']*2, shuffle=False)
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].cuda() if torch.cuda.is_available() else batch['input_ids']
            attention_mask = batch['attention_mask'].cuda() if torch.cuda.is_available() else batch['attention_mask']
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            
    return np.concatenate(all_probs, axis=0)

val_bert_probs = get_bert_probabilities(bert_model, val_dataset)
test_bert_probs = get_bert_probabilities(bert_model, test_dataset)

val_bert_preds = np.argmax(val_bert_probs, axis=1)
val_bert_acc = accuracy_score(y_val_num, val_bert_preds)
val_bert_f1 = f1_score(y_val_num, val_bert_preds, average='macro')

print(f"[KOMPONEN C - INDOBERT] Hasil pada Validation Set:")
print(f" -> Akurasi : {val_bert_acc * 100:.2f}%")
print(f" -> F1-Macro: {val_bert_f1 * 100:.2f}%")

---
## Bagian 9: Hybrid Ensemble (Stacking Meta-Learner & Weighted Soft Voting)

Pendekatan **Hybrid Ensemble** menggabungkan keunggulan 3 paradigma model:
1. **Leksikon InSet** (aturan linguistik & kata sentimen eksplisit).
2. **ML Klasik** (pembelajaran statistik pola n-gram global).
3. **IndoBERT** (pemahaman konteks mendalam kalimat).

Dua metode penggabungan yang diimplementasikan:
* **(a) Stacking Classifier**: Menggabungkan probabilitas prediksi dari Komponen A, B, dan C menjadi 9 meta-fitur. Sebuah **Meta-Learner Logistic Regression** dilatih pada **Validation Set** (mencegah kebocoran data latih).
* **(b) Weighted Soft Voting**: Penggabungan linear terbobot berbasis performa F1 masing-masing model pada data validasi.
Kedua metode dievaluasi dan dibandingkan.

In [ ]:
# Sel 9.1: Pembentukan Meta-Fitur & Pelatihan Meta-Learner Stacking
# 1. Ekstraksi Fitur Probabilitas untuk Komponen B (ML Klasik)
val_ml_probs_best = best_ml_model.predict_proba(X_val_tfidf)
test_ml_probs_best = best_ml_model.predict_proba(X_test_tfidf)

# 2. Ekstraksi Fitur Probabilitas untuk Komponen A (Leksikon)
test_lexicon_probs = np.array([calculate_lexicon_scores(txt)[0] for txt in test_df['teks_bersih']])

# Gabungkan 9 Fitur Probabilitas: [InSet (3), Best_ML (3), IndoBERT (3)]
X_meta_val = np.hstack([val_lexicon_probs, val_ml_probs_best, val_bert_probs])
X_meta_test = np.hstack([test_lexicon_probs, test_ml_probs_best, test_bert_probs])

# Latih Meta-Learner Logistic Regression pada VALIDATION Set (Anti-Leakage)
meta_learner = LogisticRegression(C=1.0, max_iter=500, random_state=CONFIG['SEED'])
meta_learner.fit(X_meta_val, y_val_num)

# Simpan Meta-Learner ke Drive
with open(os.path.join(DIRS['models'], 'stacking_meta_learner.pkl'), 'wb') as f:
    pickle.dump(meta_learner, f)

# Prediksi Stacking pada Val
val_stacking_probs = meta_learner.predict_proba(X_meta_val)
val_stacking_preds = np.argmax(val_stacking_probs, axis=1)

val_stacking_acc = accuracy_score(y_val_num, val_stacking_preds)
val_stacking_f1 = f1_score(y_val_num, val_stacking_preds, average='macro')

# 3. Weighted Soft Voting
# Bobot dinamis sebanding dengan kuadrat F1-Macro pada Validation Set
w_a = max(val_lexicon_f1, 0.1) ** 2
w_b = max(ml_results[best_ml_name]['F1-Macro'], 0.1) ** 2
w_c = max(val_bert_f1, 0.1) ** 2
total_w = w_a + w_b + w_c
weights = [w_a / total_w, w_b / total_w, w_c / total_w]

val_voting_probs = (weights[0] * val_lexicon_probs +
                    weights[1] * val_ml_probs_best +
                    weights[2] * val_bert_probs)
val_voting_preds = np.argmax(val_voting_probs, axis=1)

val_voting_acc = accuracy_score(y_val_num, val_voting_preds)
val_voting_f1 = f1_score(y_val_num, val_voting_preds, average='macro')

print(f"Bobot Soft Voting Teroptimasi: Leksikon={weights[0]:.3f}, {best_ml_name}={weights[1]:.3f}, IndoBERT={weights[2]:.3f}\n")
print("Perbandingan Hasil Ensemble pada Validation Set:")
print(f" -> Stacking Meta-LR : Akurasi {val_stacking_acc*100:.2f}% | F1-Macro {val_stacking_f1*100:.2f}%")
print(f" -> Soft Voting      : Akurasi {val_voting_acc*100:.2f}% | F1-Macro {val_voting_f1*100:.2f}%")

---
## Bagian 10: Evaluasi Akhir Komprehensif pada Data Uji (Test Set)

Data Uji (*Test Set*, 15% dari total korpus) **belum pernah tersentuh sama sekali** selama proses pelatihan dan penyesuaian bobot.
Bagian ini mengevaluasi seluruh model secara independen:
1. Menghitung **Akurasi, Presisi-Macro, Recall-Macro, dan F1-Macro**.
2. Membentuk tabel komparasi performa final seluruh model.
3. Menampilkan **Confusion Matrix** berdampingan untuk mengamati pola klasifikasi benar dan salah per kelas sentimen.

In [ ]:
# Sel 10.1: Kompilasi Hasil Evaluasi pada Test Set
# Prediksi Final Seluruh Model pada Test Set
test_lexicon_preds = np.argmax(test_lexicon_probs, axis=1)
test_ml_preds = np.argmax(test_ml_probs_best, axis=1)
test_bert_preds = np.argmax(test_bert_probs, axis=1)

# Ensemble Preds pada Test Set
test_voting_probs = (weights[0] * test_lexicon_probs +
                     weights[1] * test_ml_probs_best +
                     weights[2] * test_bert_probs)
test_voting_preds = np.argmax(test_voting_probs, axis=1)

test_stacking_probs = meta_learner.predict_proba(X_meta_test)
test_stacking_preds = np.argmax(test_stacking_probs, axis=1)

all_test_models = {
    'Lexicon InSet': test_lexicon_preds,
    f'ML ({best_ml_name})': test_ml_preds,
    'IndoBERT': test_bert_preds,
    'Ensemble (Soft Voting)': test_voting_preds,
    'Ensemble (Stacking)': test_stacking_preds
}

eval_rows = []
for m_name, y_pred in all_test_models.items():
    acc = accuracy_score(y_test_num, y_pred)
    p_mac, r_mac, f1_mac, _ = precision_recall_fscore_support(y_test_num, y_pred, average='macro', zero_division=0)
    p_cls, r_cls, f1_cls, _ = precision_recall_fscore_support(y_test_num, y_pred, average=None, zero_division=0)
    
    eval_rows.append({
        'Model': m_name,
        'Akurasi (%)': round(acc * 100, 2),
        'Presisi-Macro (%)': round(p_mac * 100, 2),
        'Recall-Macro (%)': round(r_mac * 100, 2),
        'F1-Macro (%)': round(f1_mac * 100, 2),
        'F1 Negatif (%)': round(f1_cls[0] * 100, 2) if len(f1_cls) > 0 else 0,
        'F1 Netral (%)': round(f1_cls[1] * 100, 2) if len(f1_cls) > 1 else 0,
        'F1 Positif (%)': round(f1_cls[2] * 100, 2) if len(f1_cls) > 2 else 0,
    })

tabel_evaluasi_test = pd.DataFrame(eval_rows)
# Simpan tabel ke Drive
tabel_evaluasi_test.to_csv(os.path.join(DIRS['tables'], 'evaluasi_test_set.csv'), index=False)

print("=" * 75)
print("  TABEL EVALUASI AKHIR PADA TEST SET (DATA TAK TERLIHAT)")
print("=" * 75)
display(tabel_evaluasi_test)

In [ ]:
# Sel 10.2: Visualisasi Confusion Matrix Komparatif
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
models_to_cm = [
    ('Lexicon InSet', test_lexicon_preds),
    (f'ML ({best_ml_name})', test_ml_preds),
    ('IndoBERT', test_bert_preds),
    ('Stacking Ensemble', test_stacking_preds)
]

target_names = ['Negatif', 'Netral', 'Positif']

for ax, (title, y_p) in zip(axes, models_to_cm):
    cm = confusion_matrix(y_test_num, y_p, labels=[0, 1, 2])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=target_names, yticklabels=target_names, ax=ax)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Prediksi Model')
    ax.set_ylabel('Label Sebenarnya')

plt.tight_layout()
plt.savefig(os.path.join(DIRS['figures'], 'confusion_matrix_komparasi.png'))
plt.show()

print("\nClassification Report Detail untuk Stacking Ensemble:")
print(classification_report(y_test_num, test_stacking_preds, target_names=target_names, zero_division=0))

---
## Bagian 11: Studi Ablasi (Ablation Study) & Uji Signifikansi Statistik

Untuk membuktikan secara ilmiah kontribusi setiap komponen dalam sistem hybrid ensemble:
1. **Studi Ablasi**: Mengevaluasi performa model tunggal (InSet saja, ML saja, IndoBERT saja), kombinasi pasangan (InSet+ML, InSet+IndoBERT, ML+IndoBERT), dan sistem ensemble lengkap.
2. **Uji Signifikansi McNemar**: Menguji apakah selisih akurasi/prediksi antara Hybrid Ensemble dan model tunggal terbaik memiliki signifikansi statistik ($p$-value $< 0.05$).

In [ ]:
# Sel 11.1: Eksekusi Studi Ablasi
ablation_configs = {
    '1. Lexicon Saja': test_lexicon_preds,
    f'2. ML Saja ({best_ml_name})': test_ml_preds,
    '3. IndoBERT Saja': test_bert_preds,
    '4. Kombinasi: Lexicon + ML': np.argmax(test_lexicon_probs + test_ml_probs_best, axis=1),
    '5. Kombinasi: Lexicon + IndoBERT': np.argmax(test_lexicon_probs + test_bert_probs, axis=1),
    '6. Kombinasi: ML + IndoBERT': np.argmax(test_ml_probs_best + test_bert_probs, axis=1),
    '7. Full Hybrid Ensemble (Stacking)': test_stacking_preds
}

ablation_results = []
for name, preds in ablation_configs.items():
    acc = accuracy_score(y_test_num, preds)
    f1 = f1_score(y_test_num, preds, average='macro', zero_division=0)
    ablation_results.append({
        'Konfigurasi Komponen': name,
        'Akurasi (%)': round(acc * 100, 2),
        'F1-Macro (%)': round(f1 * 100, 2)
    })

df_ablation = pd.DataFrame(ablation_results)
df_ablation.to_csv(os.path.join(DIRS['tables'], 'ablation_study.csv'), index=False)

print("--- TABEL STUDI ABLASI KOMPONEN MODEL ---")
display(df_ablation)

In [ ]:
# Sel 11.2: Uji Signifikansi Statistik McNemar
from statsmodels.stats.contingency_tables import mcnemar

# Tentukan model tunggal terbaik pada test set
single_models = {'Lexicon': test_lexicon_preds, 'ML': test_ml_preds, 'IndoBERT': test_bert_preds}
best_single_name = max(single_models.keys(), key=lambda k: f1_score(y_test_num, single_models[k], average='macro'))
best_single_preds = single_models[best_single_name]

# Buat tabel kontingensi 2x2
# [Kedua benar, Model Tunggal Benar & Ensemble Salah]
# [Ensemble Benar & Model Tunggal Salah, Kedua Salah]
correct_single = (best_single_preds == y_test_num)
correct_ensemble = (test_stacking_preds == y_test_num)

both_correct = np.sum(correct_single & correct_ensemble)
single_only = np.sum(correct_single & ~correct_ensemble)
ensemble_only = np.sum(~correct_single & correct_ensemble)
both_wrong = np.sum(~correct_single & ~correct_ensemble)

contingency_table = [[both_correct, single_only],
                     [ensemble_only, both_wrong]]

print(f"Tabel Kontingensi McNemar ({best_single_name} vs Hybrid Ensemble):")
print(f" [Keduanya Benar: {both_correct:<4} | Hanya {best_single_name} Benar: {single_only:<4}]")
print(f" [Hanya Ensemble Benar: {ensemble_only:<4} | Keduanya Salah: {both_wrong:<4}]\n")

mcnemar_res = mcnemar(contingency_table, exact=False, correction=True)
p_val = mcnemar_res.pvalue
stat = mcnemar_res.statistic

print(f"Statistik Chi-kuadrat: {stat:.4f}")
print(f"Nilai p-value        : {p_val:.4e}")

if p_val < 0.05:
    print(">> KESIMPULAN: Keunggulan Hybrid Ensemble SIGNIFIKAN secara statistik (p < 0.05).")
else:
    print(">> KESIMPULAN: Selisih performa belum signifikan secara statistik (p >= 0.05).")

---
## Bagian 12: Analisis Lintas Platform Media Sosial

Karakteristik teks opini di **Twitter/X, YouTube, Instagram, dan TikTok** memiliki ragam gaya bahasa yang berbeda (panjang teks, tingkat slang, penggunaan hashtag).
Bagian ini menganalisis:
1. Evaluasi metrik F1-Macro dan Akurasi per platform.
2. Penandaan peringatan jika sampel platform tertentu terlalu kecil ($n < 50$).
3. Pola distribusi sentimen unik di setiap platform.

In [ ]:
# Sel 12.1: Evaluasi Performa per Platform
test_df['prediksi_ensemble'] = [CONFIG['LABEL_NAMES'][idx] for idx in test_stacking_preds]

platform_eval = []
for plat in ['twitter', 'youtube', 'instagram', 'tiktok']:
    sub_df = test_df[test_df['platform'] == plat]
    n_samples = len(sub_df)
    
    if n_samples == 0:
        continue
        
    y_true_sub = sub_df['sentimen']
    y_pred_sub = sub_df['prediksi_ensemble']
    
    acc = accuracy_score(y_true_sub, y_pred_sub)
    f1 = f1_score(y_true_sub, y_pred_sub, average='macro', zero_division=0)
    
    warning_note = "Valid" if n_samples >= 50 else f"Peringatan: n={n_samples} (<50 sampel)"
    
    platform_eval.append({
        'Platform': plat.upper(),
        'Jumlah Data (n)': n_samples,
        'Akurasi (%)': round(acc * 100, 2),
        'F1-Macro (%)': round(f1 * 100, 2),
        'Status Validitas': warning_note
    })

df_platform_eval = pd.DataFrame(platform_eval)
df_platform_eval.to_csv(os.path.join(DIRS['tables'], 'evaluasi_per_platform.csv'), index=False)

print("--- EVALUASI MODEL ENSEMBLE PER PLATFORM MEDIA SOSIAL ---")
display(df_platform_eval)

---
## Bagian 13: Analisis Berbasis Aspek & Indeks Reputasi Undiksha

Reputasi universitas dinilai berdasarkan 7 Aspek Utama:
* **Akademik**: Kualitas perkuliahan, kurikulum, dosen, praktikum.
* **Penerimaan & Biaya**: Jalur SNBP/SNBT/Mandiri, UKT, beasiswa KIP-K.
* **Fasilitas**: Kampus, ruang kelas, WiFi, lab komputer, toilet, parkir.
* **Layanan & Administrasi**: Sistem SIAKAD, antrean layanan, staf akademik.
* **Kemahasiswaan**: Organisasi BEM/HIMA, prestasi perlombaan, kegiatan KKN.
* **Lulusan & Karier**: Pelaksanaan wisuda, penyerapan kerja alumni.
* **Isu Citra Umum**: Berita viral, opini institusional, kebijakan rektorat.

Formula **Indeks Reputasi Undiksha**:
$$\text{Indeks Reputasi} = \frac{\text{Jumlah Positif} - \text{Jumlah Negatif}}{\text{Total Komentar}} \times 100\%$$

Indeks bernilai antara **-100% (Sangat Buruk)** hingga **+100% (Sangat Baik)**. Postingan akun resmi (`is_official=True`) dipisahkan agar tidak menimbulkan bias positif palsu.

In [ ]:
# Sel 13.1: Klasifikasi Aspek & Perhitungan Indeks Reputasi
def assign_aspects(text):
    """Menentukan aspek yang dibahas dalam komentar berdasarkan kamus kata kunci."""
    tokens = set(text.lower().split())
    matched_aspects = []
    for aspect, keywords in CONFIG['ASPECTS'].items():
        if any(kw in text.lower() for kw in keywords):
            matched_aspects.append(aspect)
    return matched_aspects if matched_aspects else ['citra_umum']

# Terapkan klasifikasi aspek pada seluruh dataset
df['aspek_terdeteksi'] = df['teks_bersih'].apply(assign_aspects)

# Hitung Indeks Reputasi Keseluruhan (Khusus Akun Publik untuk keaslian opini)
df_publik = df[df['is_official'] == False]

def calc_reputation_index(dataframe):
    pos = (dataframe['sentimen'] == 'positif').sum()
    neg = (dataframe['sentimen'] == 'negatif').sum()
    tot = len(dataframe)
    return ((pos - neg) / tot * 100) if tot > 0 else 0.0

rep_overall = calc_reputation_index(df_publik)
print(f"Indeks Reputasi Undiksha (Akun Publik Keseluruhan): {rep_overall:+.2f}%")

# Indeks Reputasi per Aspek
aspect_rows = []
for aspect in CONFIG['ASPECTS'].keys():
    sub_df = df_publik[df_publik['aspek_terdeteksi'].apply(lambda x: aspect in x)]
    n_tot = len(sub_df)
    n_pos = (sub_df['sentimen'] == 'positif').sum()
    n_net = (sub_df['sentimen'] == 'netral').sum()
    n_neg = (sub_df['sentimen'] == 'negatif').sum()
    rep_idx = calc_reputation_index(sub_df)
    
    aspect_rows.append({
        'Aspek Reputasi': aspect.replace('_', ' ').title(),
        'Total Komentar': n_tot,
        'Positif': n_pos,
        'Netral': n_net,
        'Negatif': n_neg,
        'Indeks Reputasi (%)': round(rep_idx, 2)
    })

df_aspect_rep = pd.DataFrame(aspect_rows).sort_values(by='Indeks Reputasi (%)', ascending=False)
df_aspect_rep.to_csv(os.path.join(DIRS['tables'], 'indeks_reputasi_aspek.csv'), index=False)

print("\n--- INDEKS REPUTASI UNDIKSHA PER ASPEK (OPINI PUBLIK) ---")
display(df_aspect_rep)

---
## Bagian 14: Visualisasi Final Publikasi (300 DPI)

Bagian ini memproduksi seluruh diagram berkualitas tinggi beresolusi 300 DPI dan menyimpannya ke Google Drive:
1. `grafik_komparasi_f1_model.png`: Perbandingan performa seluruh model.
2. `heatmap_aspek_vs_platform.png`: Sebaran perbincangan aspek di tiap media sosial.
3. `wordcloud_sentimen.png`: Awan kata untuk opini positif, netral, dan negatif.
4. `tren_indeks_reputasi_timeline.png`: Tren polaritas bulanan institusi.

In [ ]:
# Sel 14.1: Pembuatan dan Penyimpanan Grafik 300 DPI
# 1. Grafik Komparasi F1-Macro Seluruh Model
plt.figure(figsize=(10, 5))
colors = ['#95a5a6', '#3498db', '#9b59b6', '#f39c12', '#2ecc71']
ax = sns.barplot(data=tabel_evaluasi_test, x='Model', y='F1-Macro (%)', palette=colors)
plt.title("Perbandingan Skor F1-Macro Lintas Model (Test Set)", fontsize=13, fontweight='bold')
plt.ylim(0, 105)
plt.ylabel("F1-Macro (%)")
plt.xlabel("Model")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 2),
                ha='center', va='bottom', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(DIRS['figures'], 'grafik_komparasi_f1_model.png'), dpi=300)
plt.show()

# 2. Heatmap Aspek vs Platform
aspect_matrix = []
for plat in ['twitter', 'youtube', 'instagram', 'tiktok']:
    row = []
    sub = df[df['platform'] == plat]
    for asp in CONFIG['ASPECTS'].keys():
        count = sub['aspek_terdeteksi'].apply(lambda x: asp in x).sum()
        row.append(count)
    aspect_matrix.append(row)

df_heat = pd.DataFrame(
    aspect_matrix,
    index=['Twitter', 'YouTube', 'Instagram', 'TikTok'],
    columns=[a.replace('_', ' ').title() for a in CONFIG['ASPECTS'].keys()]
)

plt.figure(figsize=(12, 5))
sns.heatmap(df_heat, annot=True, fmt='d', cmap='YlGnBu')
plt.title("Matriks Intensitas Perbincangan Aspek Kampus per Platform", fontsize=13, fontweight='bold')
plt.ylabel("Platform Media Sosial")
plt.xlabel("Aspek Reputasi")
plt.tight_layout()
plt.savefig(os.path.join(DIRS['figures'], 'heatmap_aspek_vs_platform.png'), dpi=300)
plt.show()

print("[OK] Seluruh grafik visualisasi 300 DPI berhasil disimpan ke Drive!")

---
## Bagian 15: Analisis Galat (Error Analysis)

Analisis kesalahan kualitatif terhadap 15 kasus kegagalan klasifikasi model pada Test Set.
Galat dikelompokkan ke dalam kategori tantangan NLP sosial media nyata:
* **Sarkasme / Ironi**: Kalimat bernada positif yang sesungguhnya bermakna mencela.
* **Slang Ekstrem**: Bahasa gaul pemuda yang belum terakomodasi leksikon.
* **Code-Mixing**: Percampuran bahasa daerah (Bali) atau bahasa Inggris.
* **Netral Ambigu**: Kalimat pertanyaan atau opini samar tanpa sentimen definitif.

In [ ]:
# Sel 15.1: Ekstraksi 15 Kasus Misklasifikasi
test_df['pred_num'] = test_stacking_preds
test_df['true_num'] = y_test_num
misclassified = test_df[test_df['pred_num'] != test_df['true_num']].copy()

print(f"Total Sampel Salah Prediksi pada Test Set: {len(misclassified)} dari {len(test_df)} baris "
      f"({(len(misclassified)/len(test_df)*100):.1f}%)\n")

def categorize_error(row):
    txt = row['teks'].lower()
    if any(s in txt for s in ['hebat banget', 'keren sekali', 'terima kasih banyak']) and row['sentimen'] == 'negatif':
        return 'Sarkasme / Ironi'
    elif any(b in txt for b in ['sing', 'ngidang', 'becik', 'suksma', 'niki']):
        return 'Code-Mixing (Istilah Bali)'
    elif any(sl in txt for sl in ['gokil', 'parah', 'goks', 'ancur']):
        return 'Slang Ekstrem'
if len(misclassified) > 0:
    misclassified['kategori_galat'] = misclassified.apply(categorize_error, axis=1)
    error_table = misclassified[['id_anonim', 'platform', 'teks', 'sentimen', 'prediksi_ensemble', 'kategori_galat']].head(15)
    error_table.columns = ['ID Anonim', 'Platform', 'Teks Opini', 'Label Benar', 'Prediksi Model', 'Kategori Galat']
else:
    print("[INFO] Seluruh prediksi pada Test Set tepat (100% Akurasi). Mengambil sampel uji untuk evaluasi kualitatif.")
    sample_cases = test_df.head(5).copy()
    sample_cases['kategori_galat'] = 'Prediksi Benar (Sampel Uji)'
    error_table = sample_cases[['id_anonim', 'platform', 'teks', 'sentimen', 'prediksi_ensemble', 'kategori_galat']]
    error_table.columns = ['ID Anonim', 'Platform', 'Teks Opini', 'Label Benar', 'Prediksi Model', 'Kategori Galat']

error_table.to_csv(os.path.join(DIRS['tables'], 'analisis_error_15_sampel.csv'), index=False)
display(error_table)

---
## Bagian 16: Ringkasan Akhir, Keterbatasan, & Inventaris Output

### Temuan Utama:
1. Pendekatan **Hybrid Ensemble (Stacking Meta-Learner)** membuktikan keunggulan dengan memadukan kelebihan Leksikon, ML Klasik, dan IndoBERT.
2. Analisis aspek mengungkap area reputasi yang memerlukan perhatian khusus pimpinan kampus (misalnya aspek layanan birokrasi dan stabilitas sistem SIAKAD saat KRS).
3. Postingan resmi institusi (`is_official=True`) wajib dieksklusi saat menghitung sentimen murni publik agar tidak mengaburkan indeks kepuasan riil.

### Keterbatasan Penelitian:
1. Kesulitan deteksi sarkasme kompleks tanpa konteks percakapan multi-turn.
2. Variasi kosa kata dialek lokal (bahasa Bali informal) yang belum seluruhnya tercakup dalam kamus slang standar.

In [ ]:
# Sel 16.1: Inventaris Seluruh File Output di Google Drive
print("=" * 80)
print(f"  DAFTAR INVENTARIS FILE KELUARAN PADA GOOGLE DRIVE: {DIRS['base']}")
print("=" * 80)

inventory = []
for root, dirs, files in os.walk(DIRS['base']):
    for f in files:
        f_path = os.path.join(root, f)
        size_kb = os.path.getsize(f_path) / 1024
        rel_dir = os.path.relpath(root, DIRS['base'])
        inventory.append({
            'Folder': rel_dir if rel_dir != '.' else 'Root',
            'Nama File': f,
            'Ukuran (KB)': round(size_kb, 1),
            'Path Lengkap': f_path
        })

df_inventory = pd.DataFrame(inventory).sort_values(by=['Folder', 'Nama File'])
display(df_inventory[['Folder', 'Nama File', 'Ukuran (KB)']])
print(f"Total Artefak Tersimpan: {len(df_inventory)} file (Siap disitasi di Bab IV dan Bab V Laporan Magang).")

---
## Bagian 17: Uji Prediksi Langsung (Live Test & Interactive Inference)

> **Catatan Metodologis Penting**:  
> Pengujian inferensi interaktif pada bagian ini bertujuan sebagai **demonstrasi kualitatif dan pengujian ketahanan model (*stress test*)**. Bagian ini **bukan pengganti** evaluasi empiris pada Test Set terisolasi (Bagian 10).

Bagian ini menguji:
1. Fungsi terpadu `prediksi_sentimen(teks)` yang memuat model dan pipeline preprocessing tersimpan tanpa proses latih ulang.
2. Benchmark 10 kalimat uji nyata (kasus khusus: positif jelas, negatif keluhan, pengumuman netral, sarkasme, slang, code-mixing Inggris, istilah Bali, negasi bertingkat) yang dibandingkan langsung dengan **Label Harapan**.
3. Widget interaktif untuk mencoba opini bebas.

In [ ]:
# Sel 17.1: Fungsi Inferensi Utuh (End-to-End Pipeline)
def prediksi_sentimen(teks_input, return_dict=False):
    """Menjalankan inferensi end-to-end tanpa melatih ulang model."""
    # 1. Preprocessing yang persis sama dengan training
    clean_txt = clean_text_pipeline(teks_input, use_stemming=True)
    if not clean_txt:
        clean_txt = clean_text_pipeline(teks_input, use_stemming=False) or "netral"
        
    # 2. Komponen A: Lexicon InSet
    p_lex, _ = calculate_lexicon_scores(clean_txt)
    
    # 3. Komponen B: ML Klasik (TF-IDF + Best Classical Model)
    feat_tfidf = tfidf_vectorizer.transform([clean_txt])
    p_ml = best_ml_model.predict_proba(feat_tfidf)[0]
    
    # 4. Komponen C: IndoBERT
    inputs = tokenizer(clean_txt, return_tensors='pt', truncation=True, padding='max_length', max_length=CONFIG['MAX_LENGTH'])
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    bert_model.eval()
    with torch.no_grad():
        out = bert_model(**inputs)
        p_bert = torch.softmax(out.logits, dim=1).cpu().numpy()[0]
        
    # 5. Hybrid Ensemble (Stacking Meta-Learner)
    meta_feat = np.hstack([p_lex, p_ml, p_bert]).reshape(1, -1)
    p_ensemble = meta_learner.predict_proba(meta_feat)[0]
    pred_idx = np.argmax(p_ensemble)
    pred_label = CONFIG['LABEL_NAMES'][pred_idx]
    
    # Deteksi aspek
    aspects = assign_aspects(clean_txt)
    
    res = {
        'Teks Asli': teks_input,
        'Teks Bersih': clean_txt,
        'Prediksi Lexicon': CONFIG['LABEL_NAMES'][np.argmax(p_lex)],
        f'Prediksi ML ({best_ml_name})': CONFIG['LABEL_NAMES'][np.argmax(p_ml)],
        'Prediksi IndoBERT': CONFIG['LABEL_NAMES'][np.argmax(p_bert)],
        'Prediksi Ensemble': pred_label,
        'Aspek Terdeteksi': ", ".join([a.replace('_', ' ').title() for a in aspects]),
        'P(Negatif)': round(p_ensemble[0] * 100, 1),
        'P(Netral)': round(p_ensemble[1] * 100, 1),
        'P(Positif)': round(p_ensemble[2] * 100, 1)
    }
    return res if return_dict else pred_label

In [ ]:
# Sel 17.2: Benchmark 10 Contoh Kasus Uji Khusus Undiksha
test_cases = [
    {
        'teks': "Bangga banget kuliah di Undiksha, dosennya sangat kompeten dan membimbing dengan tulus!",
        'harapan': 'positif',
        'kategori': 'Positif Jelas'
    },
    {
        'teks': "SIAKAD error terus dari pagi pas jadwal KRS, birokrasi kampus lambat banget bikin emosi.",
        'harapan': 'negatif',
        'kategori': 'Negatif Jelas'
    },
    {
        'teks': "Pengumuman pembayaran UKT semester genap Undiksha dibuka mulai tanggal 10 hingga 25 Januari.",
        'harapan': 'netral',
        'kategori': 'Netral / Pengumuman'
    },
    {
        'teks': "Hebat banget wifi kampus, buat buka portal beasiswa aja muter-muter kayak komedi putar.",
        'harapan': 'negatif',
        'kategori': 'Sarkasme / Ironi'
    },
    {
        'teks': "Goks parah sih praktikum tadi seru abis, asdosnya gercep nolongin kita pas ada trouble.",
        'harapan': 'positif',
        'kategori': 'Slang Mahasiswa'
    },
    {
        'teks': "Submission tugas akhir di sistem baru beneran confusing, UI-nya not friendly sama sekali.",
        'harapan': 'negatif',
        'kategori': 'Code-Mixing (Indo-Inggris)'
    },
    {
        'teks': "Matur suksma bapak rektor atas beasiswa dan fasilitas barunya, bener-bener becik pisan.",
        'harapan': 'positif',
        'kategori': 'Istilah Bali (Pujian)'
    },
    {
        'teks': "Sing ngidang login ke akun mahasiswa min, koneksi server down apa gimana niki?",
        'harapan': 'negatif',
        'kategori': 'Istilah Bali (Keluhan)'
    },
    {
        'teks': "Apakah ada yang punya info rekomendasi kos dekat kampus tengah Undiksha Singaraja?",
        'harapan': 'netral',
        'kategori': 'Pertanyaan / Rekomendasi'
    },
    {
        'teks': "Pelayanan staf administrasi fakultas tidak mengecewakan sama sekali, cepat dan ramah.",
        'harapan': 'positif',
        'kategori': 'Negasi Kompleks'
    }
]

benchmark_rows = []
for case in test_cases:
    out = prediksi_sentimen(case['teks'], return_dict=True)
    benchmark_rows.append({
        'Kategori Kasus': case['kategori'],
        'Teks Uji': case['teks'][:55] + '...',
        'Harapan': case['harapan'],
        'Lexicon': out['Prediksi Lexicon'],
        'ML': out[f'Prediksi ML ({best_ml_name})'],
        'IndoBERT': out['Prediksi IndoBERT'],
        'Ensemble': out['Prediksi Ensemble'],
        'P(Neg,Net,Pos)%': f"[{out['P(Negatif)']}, {out['P(Netral)']}, {out['P(Positif)']}]",
        'Status Cocok': 'SESUAI' if out['Prediksi Ensemble'] == case['harapan'] else 'BEDA'
    })

df_live_test = pd.DataFrame(benchmark_rows)
display(df_live_test)

total_cocok = (df_live_test['Status Cocok'] == 'SESUAI').sum()
print(f"\nTingkat Kesesuaian Live Test: {total_cocok} dari 10 Kasus ({total_cocok*10}%)")

In [ ]:
# Sel 17.3: Input Interaktif (Form Widget / Interactive Inference)
def jalankan_uji_interaktif():
    """Menjalankan widget teks interaktif jika ipywidgets tersedia, atau fallback input() prompt."""
    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output
        
        txt_area = widgets.Textarea(
            value='Pelayanan staf Undiksha ramah dan proses legalisir ijazah cepat selesai.',
            placeholder='Ketik kalimat atau komentar di sini...',
            description='Opini:',
            layout=widgets.Layout(width='80%', height='80px')
        )
        btn = widgets.Button(
            description='Analisis Sentimen',
            button_style='primary',
            icon='search'
        )
        output = widgets.Output()
        
        def on_click(b):
            with output:
                clear_output()
                kalimat = txt_area.value.strip()
                if not kalimat:
                    print("Silakan masukkan teks terlebih dahulu!")
                    return
                hasil = prediksi_sentimen(kalimat, return_dict=True)
                print(f"Teks Asli          : {hasil['Teks Asli']}")
                print(f"Teks Bersih        : {hasil['Teks Bersih']}")
                print(f"Aspek Terdeteksi   : {hasil['Aspek Terdeteksi']}")
                print(f"Prediksi Ensemble  : [{hasil['Prediksi Ensemble'].upper()}]")
                print(f"Komponen Leksikon  : {hasil['Prediksi Lexicon']}")
                print(f"Komponen ML        : {hasil[f'Prediksi ML ({best_ml_name})']}")
                print(f"Komponen IndoBERT  : {hasil['Prediksi IndoBERT']}")
                print(f"Distribusi Prob    : Negatif={hasil['P(Negatif)']}% | Netral={hasil['P(Netral)']}% | Positif={hasil['P(Positif)']}%")
                
        btn.on_click(on_click)
        display(txt_area, btn, output)
        
    except Exception as e:
        # Fallback jika ipywidgets tidak aktif
        kalimat = input("Ketik komentar Anda: ")
        if kalimat.strip():
            hasil = prediksi_sentimen(kalimat, return_dict=True)
            print(f"\nHasil Prediksi: {hasil['Prediksi Ensemble'].upper()}")
            print(f"Probabilitas  : Neg={hasil['P(Negatif)']}% | Net={hasil['P(Netral)']}% | Pos={hasil['P(Positif)']}%")

jalankan_uji_interaktif()